# Basal Melt GeoZarr Conversion

## Setup

The main thing to to do generate geozarr from a zarr store is to create multiple versions of the data at different scales and appropariately manage the structure of the resulting pyramid data tree and pusing data, metadata thought it.
In this case we are using xarayy directly, but you can do this with `nbformat` , `pyresample` and other libraries too. We are using open layers again for the visualisations.
Check the GeoZarr specification website for more etools, examples and other datasets - https://geozarr.org/ . We are using teh `geozarr-toolkit` to verify the zarr tree metadata .

In [1]:
from pathlib import Path
import shutil
import zarr

import numcodecs
import ndpyramid
import numpy as np
import pyproj
import rioxarray
import xarray as xr

from geozarr_toolkit import (
    MultiscalesConventionMetadata,
    ProjConventionMetadata,
    SpatialConventionMetadata,
    create_geozarr_attrs,
    create_multiscales_layout,
    create_zarr_conventions,
    validate_attrs,
)

ROOT = Path.cwd()
if not (ROOT / "downloaded_data").exists():
    ROOT = ROOT.parent

ZARR_INPUT = ROOT / "downloaded_data/basal_melt_map_racmo_firn_air_corrected_cog.zarr"
XARRAY_STORE = ROOT / "downloaded_data/basal_melt_geozarr.zarr"
NDPYRAMID_STORE = ROOT / "downloaded_data/basal_melt_geozarr_ndpyramid.zarr"

NAME = "basal_melt"
CHUNK = 512

## Open The Raster

`rioxarray` reads the ZARR_INPUT and gives us the raster values, projected `x` and `y` coordinates, CRS, transform, bounds, and encoded nodata value. With `masked=True`, raster nodata becomes `NaN` while we work in memory.

In [2]:
# 1. Open the Zarr store and isolate the target DataArray
ds = xr.open_zarr(ZARR_INPUT)
source = ds[NAME]

source.attrs["units"] = "m yr-1"

# 2. In Zarr/CF conventions, nodata is typically stored in the _FillValue attribute
nodata = source.attrs.get("_FillValue", np.nan)

# 3. Ensure rioxarray recognizes the spatial dimensions
source = source.rio.set_spatial_dims(x_dim="x", y_dim="y")

# If rioxarray didn't automatically pick up the CRS from the spatial_ref variable, link it manually
if source.rio.crs is None and "spatial_ref" in ds:
    crs_wkt = ds["spatial_ref"].attrs.get("spatial_ref") or ds["spatial_ref"].attrs.get("crs_wkt")
    if crs_wkt:
        source.rio.write_crs(crs_wkt, inplace=True)

# Now we can safely use the rioxarray accessor methods
source_transform = source.rio.transform()
source_bounds = list(source.rio.bounds())
source_crs = source.rio.crs

{
    "file": str(ZARR_INPUT),
    "shape": list(source.shape),
    "crs": source_crs.to_string() if source_crs else None,
    "bounds": source_bounds,
    "transform": [
        source_transform.a,
        source_transform.b,
        source_transform.c,
        source_transform.d,
        source_transform.e,
        source_transform.f,
    ],
    "encoded_nodata": float(nodata) if not np.isnan(nodata) else None,
}

{'file': '/home/krasen/tiling_test/downloaded_data/basal_melt_map_racmo_firn_air_corrected_cog.zarr',
 'shape': [6435, 6120],
 'crs': 'EPSG:3031',
 'bounds': [-3060277.25, -3217740.25, 3059722.75, 3217259.75],
 'transform': [1000.0, 0.0, -3060277.25, 0.0, -1000.0, 3217259.75],
 'encoded_nodata': None}

## Native Xarray & Zarr Writer

These helpers handle the spatial metadata calculations, while write_geozarr_native relies entirely on xarray.to_zarr and zarr to write the groups, chunks (compressed with Zstandard via numcodecs), coordinate arrays, and GeoZarr conventions.

In [3]:
def transform_from_coords(da):
    x = da.x.values
    y = da.y.values
    dx = float(x[1] - x[0])
    dy = float(y[1] - y[0])
    return [dx, 0.0, float(x[0] - dx / 2), 0.0, dy, float(y[0] - dy / 2)]


def bounds_from_transform(transform, shape):
    height, width = shape
    left = transform[2]
    top = transform[5]
    right = left + transform[0] * width
    bottom = top + transform[4] * height
    return [min(left, right), min(bottom, top), max(left, right), max(bottom, top)]


In [4]:
def transform_from_coords(da):
    x = da.x.values
    y = da.y.values
    dx = float(x[1] - x[0])
    dy = float(y[1] - y[0])
    return [dx, 0.0, float(x[0] - dx / 2), 0.0, dy, float(y[0] - dy / 2)]


def bounds_from_transform(transform, shape):
    height, width = shape
    left = transform[2]
    top = transform[5]
    right = left + transform[0] * width
    bottom = top + transform[4] * height
    return [min(left, right), min(bottom, top), max(left, right), max(bottom, top)]

def write_geozarr_native(store, levels, crs, resampling_method):
    store_path = Path(store)
    if store_path.exists():
        shutil.rmtree(store_path, ignore_errors=True)

    # Initialize the root Zarr V3 group natively
    root_group = zarr.group(store=store_path, zarr_format=3)

    level_descriptions = []
    
    for level, item in enumerate(levels):
        da = item["data"]
        factor = item["factor"]

        # Ensure types and explicit nodata replacements
        da = da.fillna(nodata).astype("float32")
        da.attrs.update({
            "_FillValue": nodata,
            "units": "m yr-1",
        })

        # Convert to Dataset and attach CRS natively
        ds = da.to_dataset(name=NAME)
        ds.rio.write_crs(crs, inplace=True)

        # Rechunk the Dask array to exactly match the target Zarr chunks
        # This prevents the overlapping chunks ValueError
        ds = ds.chunk({"x": CHUNK, "y": CHUNK})

        # Setup compression using native Zarr v3 codecs
        compressor = zarr.codecs.ZstdCodec(level=3)
        encoding = {
            NAME: {"compressors": [compressor], "chunks": (CHUNK, CHUNK)},
            "x": {"compressors": [compressor]},
            "y": {"compressors": [compressor]}
        }

        # Let Xarray handle chunking, byte-encoding, and writing
        ds.to_zarr(
            store_path,
            group=f"{NAME}/{level}",
            encoding=encoding,
            zarr_format=3,
            compute=True,
            consolidated=True
        )

        level_descriptions.append({
            "asset": f"{NAME}/{level}",
            "derived_from": f"{NAME}/0" if level else None,
            "transform": {"scale": [factor, factor]} if level else None,
            "resampling_method": resampling_method if level else None,
        })

    # Prepare GeoZarr layout and attributes using the toolkit
    layout = []
    for item in level_descriptions:
        layout_item = {"asset": item["asset"]}
        if item["derived_from"]:
            layout_item["derived_from"] = item["derived_from"]
            layout_item["transform"] = item["transform"]
            layout_item["resampling_method"] = item["resampling_method"]
        layout.append(layout_item)

    first = levels[0]["data"]
    spatial_transform = transform_from_coords(first)
    spatial_bbox = bounds_from_transform(spatial_transform, first.shape)

    root_attrs = create_geozarr_attrs(
        ["y", "x"],
        crs=crs.to_string(),
        transform=spatial_transform,
        bbox=spatial_bbox,
        shape=list(first.shape),
    )
    root_attrs.update(create_multiscales_layout(layout, resampling_method=resampling_method))
    root_attrs["zarr_conventions"] = create_zarr_conventions(
        SpatialConventionMetadata(),
        ProjConventionMetadata(),
        MultiscalesConventionMetadata(),
    )
    root_attrs["title"] = f"{NAME} GeoZarr pyramid"
    root_attrs["source_file"] = str(ZARR_INPUT)
    root_attrs["viewer"] = {
        "array": NAME,
        "tile_size": CHUNK,
        "nodata": nodata,
        "bounds": spatial_bbox,
        "levels": [
            {"path": item["asset"], "shape": list(levels[i]["data"].shape)}
            for i, item in enumerate(level_descriptions)
        ],
    }

    # Write constructed layout metadata to the root level
    root_group.attrs.update(root_attrs)

    # Replicate the required GeoZarr multiscales flag at the array group level
    array_group = root_group.require_group(NAME)
    array_group.attrs["multiscales"] = root_attrs["multiscales"]

    return root_attrs

## Xarray Transformation

The first pyramid keeps the raster in its native Antarctic Polar Stereographic CRS. Level 0 is the original ZARR_INPUT data; the following levels are simple block means made with `xarray.coarsen`.

In [5]:
xarray_levels = []

for factor in [1, 2, 4, 8, 16]:
    if factor == 1:
        level = source.astype("float32")
    else:
        level = source.coarsen(y=factor, x=factor, boundary="trim").mean(skipna=True).astype("float32")
    level.name = NAME
    xarray_levels.append({"factor": factor, "data": level})

[
    {"level": i, "factor": item["factor"], "shape": list(item["data"].shape)}
    for i, item in enumerate(xarray_levels)
]

[{'level': 0, 'factor': 1, 'shape': [6435, 6120]},
 {'level': 1, 'factor': 2, 'shape': [3217, 3060]},
 {'level': 2, 'factor': 4, 'shape': [1608, 1530]},
 {'level': 3, 'factor': 8, 'shape': [804, 765]},
 {'level': 4, 'factor': 16, 'shape': [402, 382]}]

## Xarray GeoZarr Output And Validation

The GeoZarr metadata is written at the root of the store. The validation call below is from `geozarr-toolkit`; an empty list for each convention means that convention passed.

In [6]:
xarray_attrs = write_geozarr_native(XARRAY_STORE, xarray_levels, source_crs, "xarray.coarsen.mean")

/home/krasen/tiling_test/.pixi/envs/default/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/home/krasen/tiling_test/.pixi/envs/default/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/home/krasen/tiling_test/.pixi/envs/default/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/home/krasen/tiling_test/.pixi/envs/default/lib/python3.14/site-packages/zarr/api/asynchronous.py

In [7]:
# from the geozarr toolkit examples 
from geozarr_toolkit import detect_conventions, validate_group
from zarr.storage import LocalStore

zarr_store = LocalStore(XARRAY_STORE)

root = zarr.open_group(zarr_store, mode="r")

detected = detect_conventions(dict(root.attrs))
print(f"Detected conventions: {detected}")

results = validate_group(root)
for conv, errors in results.items():
    status = "PASS" if not errors else "FAIL"
    print(f"  [{status}] {conv}")
    for err in errors:
        print(f"         {err}")

print(f"\nStore tree:")
root.tree()


Detected conventions: ['spatial', 'proj', 'multiscales']
  [PASS] spatial
  [PASS] proj
  [PASS] multiscales
  [PASS] zarr_conventions

Store tree:


/
└── basal_melt
    ├── 0
    │   ├── basal_melt (6435, 6120) float32
    │   ├── spatial_ref () int64
    │   ├── x (6120,) float64
    │   └── y (6435,) float64
    ├── 1
    │   ├── basal_melt (3217, 3060) float32
    │   ├── spatial_ref () int64
    │   ├── x (3060,) float64
    │   └── y (3217,) float64
    ├── 2
    │   ├── basal_melt (1608, 1530) float32
    │   ├── spatial_ref () int64
    │   ├── x (1530,) float64
    │   └── y (1608,) float64
    ├── 3
    │   ├── basal_melt (804, 765) float32
    │   ├── spatial_ref () int64
    │   ├── x (765,) float64
    │   └── y (804,) float64
    └── 4
        ├── basal_melt (402, 382) float32
        ├── spatial_ref () int64
        ├── x (382,) float64
        └── y (402,) float64

In [12]:
xr.open_zarr(XARRAY_STORE, group='basal_melt/3', consolidated=False)

<xarray.Dataset> Size: 2MB
Dimensions:      (y: 804, x: 765)
Coordinates:
  * y            (y) float64 6kB 3.213e+06 3.205e+06 ... -3.203e+06 -3.211e+06
  * x            (x) float64 6kB -3.056e+06 -3.048e+06 ... 3.048e+06 3.056e+06
Data variables:
    basal_melt   (y, x) float32 2MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    spatial_ref  int64 8B ...

### Visualise the data

In [ ]:
# from pathlib import Path
# from html import escape
# from IPython.display import HTML, display

# candidate_paths = [
#     Path("geozarr_viewer.html"),
#     Path("4_Visualisation/geozarr_viewer.html"),
# ]

# html_path = next((path for path in candidate_paths if path.exists()), None)
# if html_path is None:
#     raise FileNotFoundError("Could not find cog_openlayers.html")

# html_code = html_path.read_text(encoding="utf-8")

# display(HTML(f"""
# <iframe
#     srcdoc="{escape(html_code)}"
#     style="width: 100%; height: 850px; border: 0;"
#     loading="eager">
# </iframe>
# """))